In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import tensorflow as tf
from tensorflow import keras


In [ ]:
project_dir = "/content/drive/MyDrive/météo_dataset/saved_model_v1"


In [ ]:
keras_model_path = os.path.join(project_dir, "weather_mlp_6h.keras")
tflite_model_path = os.path.join(project_dir, "weather_mlp_6h.tflite")
header_file_path = os.path.join(project_dir, "model_data.h")

In [ ]:
model = keras.models.load_model(keras_model_path)

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,845 (15.02 KB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,564 (10.02 KB)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open(tflite_model_path, "wb") as f:
    f.write(tflite_model)


Saved artifact at '/tmp/tmp8iyyn064'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 22), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  136096612436880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136096612439376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136096612439760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136096612438800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136096612438032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136096612437072: TensorSpec(shape=(), dtype=tf.resource, name=None)


In [ ]:
with open(tflite_model_path, "rb") as f:
    data = f.read()

with open(header_file_path, "w") as f:
    f.write("#ifndef MODEL_DATA_H\n")
    f.write("#define MODEL_DATA_H\n\n")
    f.write("const unsigned char model_data[] = {")
    for i, b in enumerate(data):
        if i % 12 == 0:
            f.write("\n  ")
        f.write(f"0x{b:02x}, ")
    f.write("\n};\n\n")
    f.write(f"const unsigned int model_data_len = {len(data)};\n\n")
    f.write("#endif  // MODEL_DATA_H\n")